In [7]:
from pathlib import Path
 
import numpy as np
import pandas as pd
 

In [15]:
RAW = Path("data/raw")
OUT = Path("data/processed")
MONTH = "2026-09"  # the month we are analyzing (YYYY-MM)
 
TRIPS_FILE = RAW / f"{MONTH.replace('-', '')}-bluebikes-tripdata.csv"
STATIONS_FILE = RAW / "current_bluebikes_stations.csv"
WEATHER_FILE = RAW / "noaa_boston_daily_aug.csv"
 
NEU_LAT, NEU_LON = 42.3398, -71.0892  # Northeastern, roughly Centennial Common
RADIUS_KM = 1.2  # tune this until we get 30-60 stations

In [9]:
def haversine_km(lat1, lon1, lat2, lon2):
    """Great-circle distance in km. Works on numpy arrays / pandas Series."""
    lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
    a = (np.sin((lat2 - lat1) / 2) ** 2
         + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2)
    return 6371.0 * 2 * np.arcsin(np.sqrt(a))

def pick_column(df, candidates):
    """Return the first column name in df that matches one of the candidates."""
    for c in candidates:
        if c in df.columns:
            return c
    raise KeyError(f"None of {candidates} found. Columns are: {list(df.columns)}")

In [10]:
def load_stations(path=STATIONS_FILE):
    """
    Bluebikes' station CSV sometimes has a 'Last Updated' line above the
    real header, so we search the first few lines for the header row.
    """
    with open(path, encoding="utf-8-sig") as f:
        first_lines = [next(f, "") for _ in range(10)]
    header_row = next(i for i, line in enumerate(first_lines) if "lat" in line.lower())
 
    df = pd.read_csv(path, skiprows=header_row, dtype=str, encoding="utf-8-sig")
    df.columns = df.columns.str.strip().str.lower()
 
    stations = pd.DataFrame({
        "station_id": df[pick_column(df, ["number", "station id", "short_name", "id"])].str.strip(),
        "name": df[pick_column(df, ["name", "station name"])].str.strip(),
        "lat": pd.to_numeric(df[pick_column(df, ["lat", "latitude"])], errors="coerce"),
        "lon": pd.to_numeric(df[pick_column(df, ["long", "lon", "lng", "longitude"])], errors="coerce"),
        "capacity": pd.to_numeric(df[pick_column(df, ["total docks", "total_docks", "capacity", "docks"])],
                                  errors="coerce"),
    })
    if "district" in df.columns:
        stations["district"] = df["district"].str.strip()
 
    stations = stations.dropna(subset=["lat", "lon"]).drop_duplicates("station_id")
    stations["dist_to_neu_km"] = haversine_km(NEU_LAT, NEU_LON, stations["lat"], stations["lon"])
    return stations.sort_values("dist_to_neu_km").reset_index(drop=True)
 
 
def neu_cluster(stations, radius_km=RADIUS_KM):
    return stations[stations["dist_to_neu_km"] <= radius_km].reset_index(drop=True)


 

In [11]:
def load_trips(path=TRIPS_FILE):
    """Trip files from 2024 onward (ride_id, started_at, start_station_id, ...)."""
    df = pd.read_csv(path, dtype={"start_station_id": str, "end_station_id": str},
                     low_memory=False)
    df.columns = df.columns.str.strip().str.lower()
 
    df["started_at"] = pd.to_datetime(df["started_at"], errors="coerce")
    df["ended_at"] = pd.to_datetime(df["ended_at"], errors="coerce")
    df["start_station_id"] = df["start_station_id"].str.strip()
    df["end_station_id"] = df["end_station_id"].str.strip()
    df["duration_min"] = (df["ended_at"] - df["started_at"]).dt.total_seconds() / 60
 
    n_before = len(df)
    df = df.dropna(subset=["started_at", "ended_at"])
    df = df[df["duration_min"] > 0]
    print(f"[trips] {n_before:,} rows read, {len(df):,} kept after removing bad timestamps")
    return df
 

In [12]:
def filter_trips_to_cluster(trips, cluster):
    """
    Keep a trip if it starts OR ends in the cluster: departures from cluster
    stations take bikes away, arrivals bring them in, and both matter for
    whether a station runs empty or full.
    """
    ids = set(cluster["station_id"])
    starts_in = trips["start_station_id"].isin(ids)
    ends_in = trips["end_station_id"].isin(ids)
    out = trips[starts_in | ends_in].copy()
    out["starts_in_cluster"] = starts_in[starts_in | ends_in]
    out["ends_in_cluster"] = ends_in[starts_in | ends_in]
    return out.reset_index(drop=True)

In [13]:
def check_unmatched_ids(trips, stations):
    """Trip station IDs missing from the station list (removed or renamed stations)."""
    known = set(stations["station_id"])
    trip_ids = set(trips["start_station_id"].dropna()) | set(trips["end_station_id"].dropna())
    missing = sorted(trip_ids - known)
    if missing:
        print(f"[check] {len(missing)} trip station IDs are not in the station list, "
              f"e.g. {missing[:5]}")
    return missing

In [14]:
def load_weather(path=WEATHER_FILE, month=MONTH):
    """
    NOAA Climate Data Online daily summaries (GHCN-Daily), e.g. Boston Logan
    station USW00014739. Units depend on what you chose when downloading
    (Standard = inches / deg F, Metric = mm / deg C).
    """
    df = pd.read_csv(path)
    df.columns = df.columns.str.strip().str.upper()
    df["DATE"] = pd.to_datetime(df["DATE"])
 
    keep = [c for c in ["DATE", "PRCP", "SNOW", "SNWD", "TMAX", "TMIN", "TAVG", "AWND"]
            if c in df.columns]
    df = df[keep]
    df = df[df["DATE"].dt.strftime("%Y-%m") == month].sort_values("DATE")
 
    if df.empty:
        raise ValueError(f"No weather rows for {month}. Check the file's date range.")
    return df.reset_index(drop=True)
 

In [16]:
OUT.mkdir(parents=True, exist_ok=True)

stations = load_stations()
cluster = neu_cluster(stations)
print(f"[stations] {len(stations)} total, {len(cluster)} within {RADIUS_KM} km of NEU")
print(cluster[["station_id", "name", "capacity", "dist_to_neu_km"]].to_string(index=False))

trips = load_trips()
check_unmatched_ids(trips, stations)
neu_trips = filter_trips_to_cluster(trips, cluster)
print(f"[trips] {len(neu_trips):,} trips touch the NEU cluster "
        f"({neu_trips['starts_in_cluster'].sum():,} departures, "
        f"{neu_trips['ends_in_cluster'].sum():,} arrivals)")

weather = load_weather()
print(f"[weather] {len(weather)} days loaded for {MONTH}")
print(weather.head().to_string(index=False))

cluster.to_csv(OUT / "neu_stations.csv", index=False)
neu_trips.to_csv(OUT / f"neu_trips_{MONTH}.csv", index=False)
weather.to_csv(OUT / f"weather_{MONTH}.csv", index=False)
print(f"[done] saved to {OUT}/")

KeyError: "None of ['number', 'station id', 'short_name', 'id'] found. Columns are: ['name', 'lat', 'long', 'seasonal status', 'municipality', 'total docks', 'station id (to match to historic system data)']"